# RQ1 — Table 3: full metrics over 20 predetermined re-splits

This notebook produces the complete confirmatory table for all six matched baseline–plus pairs. Each metric is reported as **mean ± SD across 20 predetermined stratified 60/20/20 re-splits**; each split-level value first averages model seeds 12–16.

The same 2,763 labeled HEIG-Phish nodes are reused across the 20 splits. This measures split robustness and is not external or temporal generalization.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_MANIFEST_SHA256 = '13a77c8927c729c66ca9f7a10bd5f32acece7e10c5b2457159048de1128a69b5'
EXPECTED_DATA_SHA256 = '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def search_roots():
    roots = [Path.cwd(), Path('/kaggle/input'), Path('/kaggle/working'), Path(r'D:/Takedo/Documents/DAP/rq1_reporting_tables_20260926')]
    return [p for p in roots if p.exists()]

def locate_by_hash(name, expected_hash):
    matches = []
    for root in search_roots():
        for path in root.rglob(name):
            try:
                if sha256_file(path) == expected_hash:
                    matches.append(path)
            except OSError:
                pass
    unique = {str(p.resolve()): p for p in matches}
    assert unique, f'Cannot find {name} with SHA-256 {expected_hash}. Upload the complete reporting bundle.'
    return next(iter(unique.values()))

manifest_path = locate_by_hash('table3_source_manifest.json', EXPECTED_MANIFEST_SHA256)
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['source_data_sha256'] == EXPECTED_DATA_SHA256
assert manifest['split_seeds'] == list(range(2001, 2021))
assert manifest['model_seeds'] == [12, 13, 14, 15, 16]
source_dir = manifest_path.parent
print('Source directory:', source_dir)

Source directory: /kaggle/input/datasets/tranminhhuy15/dasdasd/rq1_reporting_tables_final_3tables_20260926/sources/table3


In [2]:
for item in manifest['files']:
    path = source_dir / item['name']
    assert path.is_file(), path
    assert sha256_file(path) == item['sha256'], f'SHA-256 mismatch: {path}'
audit = json.loads((source_dir / 'audit_summary.json').read_text(encoding='utf-8'))
assert audit['status'] == manifest['audit_status_required']
assert audit['integrity'] == {
    'models': 6, 'splits': 120, 'score_cubes': 120, 'checkpoints': 1200,
    'prediction_rows': 663600, 'test_result_rows': 1200,
    'history_files': 600, 'early_stopping_checks': 600,
}
print('Integrity status:', audit['status'])

Integrity status: PASS — all six result packages independently recomputed and verified


In [3]:
h1 = pd.read_csv(source_dir / 'h1_six_pairs_holm.csv')
full = pd.read_csv(source_dir / 'twelve_model_full_metrics.csv')
legacy = pd.read_csv(source_dir / 'twelve_model_metrics.csv')
assert len(h1) == 6 and len(full) == 12 and len(legacy) == 12
assert set(h1['splits_plus_better'].astype(int)) == {20}
passed = h1['meets_predeclared_rule'].astype(str).str.lower().map({'true': True, 'false': False})
assert passed.all()
expected_models = ['LR', 'LR+', 'RF', 'RF+', 'LightGBM', 'LightGBM+', 'GAT-Norm', 'GAT+', 'GraphSAGE-Norm', 'GraphSAGE+', 'Meta-IFD', 'Meta-IFD+']
assert full['model'].tolist() == expected_models
check = full.set_index('model').join(legacy.set_index('model'), lsuffix='_full', rsuffix='_legacy')
for metric in ['precision_mean', 'recall_mean', 'binary_f1_mean', 'macro_f1_mean', 'pr_auc_mean', 'fpr_mean']:
    assert np.allclose(check[f'{metric}_full'], check[f'{metric}_legacy'], atol=1e-12)
for _, row in h1.iterrows():
    assert np.isclose(row['baseline_macro_f1'], full.set_index('model').loc[row['baseline'], 'macro_f1_mean'], atol=1e-12)
    assert np.isclose(row['plus_macro_f1'], full.set_index('model').loc[row['plus'], 'macro_f1_mean'], atol=1e-12)
print('All metrics and all six matched tests passed cross-checks.')

All metrics and all six matched tests passed cross-checks.


In [4]:
metric_specs = [
    ('Precision', 'precision'), ('Recall', 'recall'), ('Binary-F1', 'binary_f1'),
    ('Micro-F1', 'micro_f1'), ('Macro-F1', 'macro_f1'), ('PR-AUC', 'pr_auc'), ('FPR', 'fpr'),
]
inference = h1.set_index('plus')
numeric_rows, display_rows = [], []
for _, row in full.iterrows():
    model = row['model']
    is_plus = row['variant'] == 'plus'
    nr = {'Model': model, 'Role': 'Plus' if is_plus else 'Baseline'}
    dr = {'Model': model, 'Role': nr['Role']}
    for label, stem in metric_specs:
        mean = float(row[f'{stem}_mean']) * 100
        sd = float(row[f'{stem}_sd_across_splits']) * 100
        nr[f'{label} mean (%)'] = mean
        nr[f'{label} SD (%)'] = sd
        dr[label] = f'{mean:.2f} ± {sd:.2f}'
    if is_plus:
        stat = inference.loc[model]
        nr.update({
            'Delta Macro-F1 (pp)': float(stat['mean_delta']) * 100,
            'NB CI95 lower (pp)': float(stat['ci95_lower']) * 100,
            'NB CI95 upper (pp)': float(stat['ci95_upper']) * 100,
            'Holm-adjusted p': float(stat['p_holm']),
            'Plus wins': int(stat['splits_plus_better']),
            'Rule': 'PASS' if str(stat['meets_predeclared_rule']).lower() == 'true' else 'FAIL',
        })
        dr.update({
            'Δ Macro (pp)': f"{nr['Delta Macro-F1 (pp)']:+.2f}",
            'NB 95% CI': f"[{nr['NB CI95 lower (pp)']:+.2f}, {nr['NB CI95 upper (pp)']:+.2f}]",
            'Holm p': '<0.001' if nr['Holm-adjusted p'] < 0.001 else f"{nr['Holm-adjusted p']:.4f}",
            'Wins': f"{nr['Plus wins']}/20",
            'Rule': nr['Rule'],
        })
    else:
        nr.update({'Delta Macro-F1 (pp)': np.nan, 'NB CI95 lower (pp)': np.nan, 'NB CI95 upper (pp)': np.nan, 'Holm-adjusted p': np.nan, 'Plus wins': np.nan, 'Rule': ''})
        dr.update({'Δ Macro (pp)': '—', 'NB 95% CI': '—', 'Holm p': '—', 'Wins': '—', 'Rule': '—'})
    numeric_rows.append(nr)
    display_rows.append(dr)

table_numeric = pd.DataFrame(numeric_rows)
table_display = pd.DataFrame(display_rows)
display(table_display.style.hide(axis='index').set_caption(
    'Table 3. Full metrics and confirmatory matched inference across 20 predetermined HEIG-Phish re-splits (%)'
))

Model,Role,Precision,Recall,Binary-F1,Micro-F1,Macro-F1,PR-AUC,FPR,Δ Macro (pp),NB 95% CI,Holm p,Wins,Rule
LR,Baseline,82.59 ± 2.09,98.51 ± 0.92,89.84 ± 1.35,90.27 ± 1.41,90.25 ± 1.40,93.90 ± 1.52,16.09 ± 2.35,—,—,—,—,—
LR+,Plus,95.69 ± 0.88,97.05 ± 1.36,96.36 ± 0.76,96.81 ± 0.65,96.76 ± 0.66,98.61 ± 0.78,3.38 ± 0.73,+6.51,"[+4.81, +8.20]",<0.001,20/20,PASS
RF,Baseline,94.94 ± 1.21,95.60 ± 1.67,95.26 ± 0.96,95.85 ± 0.83,95.79 ± 0.84,98.32 ± 0.50,3.95 ± 0.99,—,—,—,—,—
RF+,Plus,98.85 ± 0.47,96.96 ± 1.17,97.89 ± 0.62,98.18 ± 0.52,98.15 ± 0.53,99.59 ± 0.27,0.87 ± 0.36,+2.36,"[+1.57, +3.15]",<0.001,20/20,PASS
LightGBM,Baseline,94.70 ± 1.26,95.87 ± 1.71,95.27 ± 0.83,95.85 ± 0.71,95.79 ± 0.73,97.64 ± 1.43,4.17 ± 1.06,—,—,—,—,—
LightGBM+,Plus,98.26 ± 0.64,97.39 ± 1.37,97.82 ± 0.82,98.11 ± 0.70,98.08 ± 0.72,99.53 ± 0.42,1.33 ± 0.49,+2.29,"[+1.28, +3.31]",<0.001,20/20,PASS
GAT-Norm,Baseline,83.09 ± 2.63,91.59 ± 2.93,87.06 ± 2.09,88.12 ± 1.93,88.03 ± 1.94,91.38 ± 2.19,14.55 ± 2.68,—,—,—,—,—
GAT+,Plus,96.52 ± 0.97,97.01 ± 1.30,96.75 ± 0.68,97.16 ± 0.58,97.12 ± 0.59,98.88 ± 0.58,2.72 ± 0.80,+9.08,"[+6.53, +11.64]",<0.001,20/20,PASS
GraphSAGE-Norm,Baseline,91.06 ± 1.96,94.57 ± 1.88,92.74 ± 1.03,93.54 ± 0.92,93.46 ± 0.93,95.85 ± 1.57,7.25 ± 1.77,—,—,—,—,—
GraphSAGE+,Plus,97.54 ± 0.98,97.84 ± 1.07,97.68 ± 0.59,97.97 ± 0.51,97.94 ± 0.52,98.98 ± 0.94,1.92 ± 0.80,+4.48,"[+3.57, +5.38]",<0.001,20/20,PASS


### Statistical definition

The delta, interval, p-value, win count and decision appear on each plus row and refer to its immediately preceding matched baseline. Nadeau–Bengio correction uses the test/train ratio 553/1657, and Holm correction covers all six primary pairs. A pair passes only when Δ Macro-F1 ≥ 0.5 percentage points, the corrected 95% CI lower bound is above zero, and Holm-adjusted p < 0.05.

In [5]:
compact = pd.DataFrame({
    'Pair': h1['baseline'] + ' → ' + h1['plus'],
    'Baseline Macro-F1 (%)': h1['baseline_macro_f1'].astype(float) * 100,
    'Plus Macro-F1 (%)': h1['plus_macro_f1'].astype(float) * 100,
    'Delta (pp)': h1['mean_delta'].astype(float) * 100,
    'CI95 lower (pp)': h1['ci95_lower'].astype(float) * 100,
    'CI95 upper (pp)': h1['ci95_upper'].astype(float) * 100,
    'Holm-adjusted p': h1['p_holm'].astype(float),
    'Plus better (splits)': h1['splits_plus_better'].astype(int),
    'Rule': np.where(passed, 'PASS', 'FAIL'),
})
display(compact.round(4).style.hide(axis='index').set_caption('Compact inferential view of Table 3'))

Pair,Baseline Macro-F1 (%),Plus Macro-F1 (%),Delta (pp),CI95 lower (pp),CI95 upper (pp),Holm-adjusted p,Plus better (splits),Rule
LR → LR+,90.251800,96.759300,6.507600,4.811200,8.204000,0.000000,20,PASS
RF → RF+,95.786400,98.147400,2.361000,1.573700,3.148300,0.000000,20,PASS
LightGBM → LightGBM+,95.785200,98.075800,2.290600,1.276100,3.305100,0.000300,20,PASS
GAT-Norm → GAT+,88.034600,97.118300,9.083700,6.529500,11.637800,0.000000,20,PASS
GraphSAGE-Norm → GraphSAGE+,93.462100,97.939500,4.477400,3.572800,5.382000,0.000000,20,PASS
Meta-IFD → Meta-IFD+,97.342200,98.357900,1.015600,0.500700,1.530600,0.000600,20,PASS


In [6]:
output_dir = Path('/kaggle/working/rq1_table_outputs/table3_full') if Path('/kaggle/working').exists() else Path.cwd() / 'rq1_table_outputs' / 'table3_full'
output_dir.mkdir(parents=True, exist_ok=True)
table_numeric.to_csv(output_dir / 'table3_full_metrics_numeric.csv', index=False)
table_display.to_csv(output_dir / 'table3_full_metrics_display.csv', index=False)
compact.to_csv(output_dir / 'table3_compact_inference.csv', index=False)

def dataframe_to_markdown(frame):
    clean = frame.astype(str).map(lambda x: x.replace('|', chr(92) + '|'))
    lines = ['| ' + ' | '.join(clean.columns) + ' |', '|' + '|'.join(['---'] * len(clean.columns)) + '|']
    lines.extend('| ' + ' | '.join(row) + ' |' for row in clean.to_numpy().tolist())
    return '\n'.join(lines) + '\n'

(output_dir / 'table3_full_metrics.md').write_text(dataframe_to_markdown(table_display), encoding='utf-8')
(output_dir / 'table3_full_metrics.tex').write_text(
    table_display.to_latex(index=False, escape=True, caption='Full metrics and confirmatory matched inference over 20 predetermined HEIG-Phish re-splits (percent).', label='tab:ufe-20split-full'),
    encoding='utf-8'
)
metadata = {
    'table': 3, 'scientific_role': manifest['scientific_role'],
    'manifest_sha256': EXPECTED_MANIFEST_SHA256, 'source_data_sha256': EXPECTED_DATA_SHA256,
    'split_seeds': manifest['split_seeds'], 'model_seeds': manifest['model_seeds'],
    'all_six_pairs_pass': bool((compact['Rule'] == 'PASS').all()),
    'limitation': 'The 20 re-splits reuse the same 2,763 labeled nodes and do not constitute external or temporal generalization.'
}
(output_dir / 'table3_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved full Table 3 artifacts to', output_dir)

Saved full Table 3 artifacts to /kaggle/working/rq1_table_outputs/table3_full
